# Clase 07 · De los documentos al índice
**Inteligencia Artificial Generativa y Sistemas Inteligentes · EAM 2026-II**

En esta práctica prepararemos dos documentos, compararemos dos formas de fragmentarlos y guardaremos un índice para volver a consultarlo. Trabajaremos en Colab, con el mismo modelo de embeddings de la clase anterior. La primera ejecución necesita Internet para instalar la biblioteca y descargar el modelo; podemos usar CPU.

Ejecuten las celdas en orden. Los procedimientos del ejemplo son **ficticios**. La consulta se escribe en el punto **5**.

## 1. Preparación
Instalamos la biblioteca y cargamos el modelo. Comprobamos su límite de entrada para evitar que el texto se recorte al generar los embeddings.

In [1]:
%pip -q install sentence-transformers==6.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 9.3 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import re, json, hashlib, zipfile
import numpy as np
from sentence_transformers import SentenceTransformer
from huggingface_hub import model_info

MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
REVISION = model_info(MODELO).sha
modelo = SentenceTransformer(MODELO, revision=REVISION)
tokenizador = modelo.tokenizer
MAX_TOKENS = 80       # Presupuesto del fragmento completo, incluido el encabezado.
SOLAPAMIENTO = 12    # Tokens repetidos entre ventanas de un mismo bloque.

def contar_tokens(texto):
    return len(tokenizador.encode(texto, add_special_tokens=True, truncation=False))

print("Límite del modelo:", modelo.max_seq_length, "tokens")
print("Presupuesto de la práctica:", MAX_TOKENS, "tokens")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Límite del modelo: 128 tokens
Presupuesto de la práctica: 80 tokens


## 2. Los documentos del corpus
Esta celda crea dos archivos Markdown en la sesión. Después los leeremos desde la carpeta, como haremos con las fuentes del proyecto. Hemos dejado algunos espacios y líneas sobrantes para observar la limpieza.

In [3]:
carpeta = Path("corpus_clase07")
carpeta.mkdir(exist_ok=True)
fuentes = {
    "acceso.md": """# Manual de acceso

## Recuperar contraseña

Para  restablecer la contraseña del campus virtual, ingrese al portal de soporte y seleccione Recuperar acceso. Escriba el correo institucional registrado y revise el mensaje que recibirá. El enlace permite establecer una contraseña nueva; la aplicación no debe pedir que comparta la contraseña anterior.


Si no puede abrir el correo institucional, contacte a soporte para verificar su identidad. En este caso no basta con repetir el envío del enlace. No se debe solicitar la contraseña por correo ni incluirla en una captura de pantalla.

## Error 403

Si aparece un error 403 al iniciar sesión, registre un incidente y adjunte una captura con el mensaje visible. Incluya la fecha y el recurso al que intentó acceder. Este error no demuestra por sí solo que la cuenta haya sido suspendida; soporte debe verificar la causa.
""",
    "constancias.md": """# Guía de solicitudes documentales

## Constancias académicas

La solicitud de una constancia académica se radica en el módulo de documentos. Indique el tipo de constancia y el correo al que desea recibirla. Antes de emitirla, se revisan el registro de matrícula y los documentos requeridos para el trámite. Si falta alguno de los documentos, la solicitud permanece pendiente de información y la constancia no se entrega. Adjuntar el documento faltante permite continuar la revisión, pero no significa que la constancia ya esté emitida.

Conserve el código asignado para consultar el avance. Si necesita corregir los datos de la solicitud, contacte al área responsable e indique ese código. Crear varias solicitudes iguales no reemplaza la corrección de la original.

## Seguimiento

El estado de una solicitud se consulta en el módulo de seguimiento mediante su código, por ejemplo SOL-1001. La consulta del estado actual requiere revisar el registro correspondiente. Este documento explica el procedimiento y no contiene los estados individuales de las solicitudes.
""",
}
for nombre, contenido in fuentes.items():
    (carpeta / nombre).write_text(contenido, encoding="utf-8")
print("Archivos:", ", ".join(fuentes))

Archivos: acceso.md, constancias.md


## 3. Lectura y limpieza
Conservamos archivo, título, versión y huella del original. Reducimos espacios y líneas repetidas; mantenemos párrafos, negaciones y condiciones. Lean la muestra y compárenla con el documento anterior.

In [4]:
documentos = []
for ruta in sorted(carpeta.glob("*.md")):
    original = ruta.read_text(encoding="utf-8")
    limpio = re.sub(r"[ \t]+", " ", original)
    limpio = re.sub(r"\n{3,}", "\n\n", limpio).strip()
    documentos.append({
        "document_id": ruta.stem, "source": ruta.name,
        "title": limpio.splitlines()[0].lstrip("# "), "version": "1.0-didactica",
        "source_hash": hashlib.sha256(original.encode("utf-8")).hexdigest(),
        "texto": limpio,
    })
    print(f"{ruta.name}: {len(original)} → {len(limpio)} caracteres")
print("\nMuestra del texto limpio:\n", documentos[0]["texto"][:450])

acceso.md: 870 → 867 caracteres
constancias.md: 1068 → 1067 caracteres

Muestra del texto limpio:
 # Manual de acceso

## Recuperar contraseña

Para restablecer la contraseña del campus virtual, ingrese al portal de soporte y seleccione Recuperar acceso. Escriba el correo institucional registrado y revise el mensaje que recibirá. El enlace permite establecer una contraseña nueva; la aplicación no debe pedir que comparta la contraseña anterior.

Si no puede abrir el correo institucional, contacte a soporte para verificar su identidad. En este c


## 4. Dos estrategias de fragmentación
**Ventanas:** recorremos cada sección por tokens, con solapamiento. **Párrafos:** conservamos sus límites y subdividimos solo los párrafos que exceden el presupuesto. En la segunda estrategia no repetimos texto entre párrafos; sí entre ventanas de un párrafo largo.

Ambas incluyen título y sección en el texto vectorizado. Conservamos el mismo modelo, presupuesto y medida de búsqueda. Revisen los tamaños y el final de los fragmentos: ¿alguna condición quedó separada de su consecuencia?

In [5]:
def ventanas(texto, limite):
    offsets = tokenizador(texto, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
    if limite <= SOLAPAMIENTO:
        raise ValueError("El encabezado deja muy poco espacio para el contenido.")
    for inicio in range(0, len(offsets), limite - SOLAPAMIENTO):
        fin = min(inicio + limite, len(offsets))
        yield texto[offsets[inicio][0]:offsets[fin - 1][1]].strip()
        if fin == len(offsets):
            break

def fragmentar(estrategia):
    registros = []
    for documento in documentos:
        for numero, bloque in enumerate(re.split(r"(?m)^## ", documento["texto"])[1:], 1):
            seccion, cuerpo = bloque.split("\n", 1)
            encabezado = documento["title"] + " / " + seccion + "\n"
            limite = MAX_TOKENS - contar_tokens(encabezado) - 2
            partes = [cuerpo] if estrategia == "ventanas" else re.split(r"\n\s*\n", cuerpo.strip())
            for parte in partes:
                for texto in ventanas(parte.strip(), limite):
                    contenido = encabezado + texto
                    tokens = contar_tokens(contenido)
                    if tokens > min(MAX_TOKENS, modelo.max_seq_length):
                        raise ValueError("Un fragmento supera el límite: ajuste el presupuesto.")
                    registros.append({
                        **{k: documento[k] for k in ["document_id", "source", "title", "version", "source_hash"]},
                        "chunk_id": f"{documento['document_id']}:{documento['version']}:{estrategia}:{len(registros)}",
                        "section": seccion, "section_number": numero, "tokens": tokens, "texto": contenido,
                    })
    return registros

fragmentos = {nombre: fragmentar(nombre) for nombre in ["ventanas", "parrafos"]}
for nombre, registros in fragmentos.items():
    print(f"\n{nombre}: {len(registros)} fragmentos; máximo {max(r['tokens'] for r in registros)} tokens")
    for registro in registros[:2]:
        print(registro["chunk_id"], "|", registro["source"], "|", registro["tokens"], "tokens")
        print(registro["texto"], "\n")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (145 > 128). Running this sequence through the model will result in indexing errors



ventanas: 7 fragmentos; máximo 78 tokens
acceso:1.0-didactica:ventanas:0 | acceso.md | 78 tokens
Manual de acceso / Recuperar contraseña
Para restablecer la contraseña del campus virtual, ingrese al portal de soporte y seleccione Recuperar acceso. Escriba el correo institucional registrado y revise el mensaje que recibirá. El enlace permite establecer una contraseña nueva; la aplicación no debe pedir que comparta la contraseña anterior.

Si no puede abrir el correo institucional, contacte a soporte 

acceso:1.0-didactica:ventanas:1 | acceso.md | 56 tokens
Manual de acceso / Recuperar contraseña
.

Si no puede abrir el correo institucional, contacte a soporte para verificar su identidad. En este caso no basta con repetir el envío del enlace. No se debe solicitar la contraseña por correo ni incluirla en una captura de pantalla. 


parrafos: 7 fragmentos; máximo 78 tokens
acceso:1.0-didactica:parrafos:0 | acceso.md | 67 tokens
Manual de acceso / Recuperar contraseña
Para restablecer la c

## 5. Consulta del usuario y comparación
Calculamos los embeddings de ambos conjuntos una vez. Después escribimos la pregunta en **CONSULTA** y comparamos los tres primeros resultados. Cada resultado conserva el archivo y la sección; vuelvan al original para comprobar su contenido. La similitud coseno ordena candidatos, no mide certeza.

In [6]:
matrices = {
    nombre: modelo.encode([r["texto"] for r in registros], normalize_embeddings=True, show_progress_bar=False)
    for nombre, registros in fragmentos.items()
}

def buscar(consulta, registros, matriz, k=3):
    if not consulta.strip():
        raise ValueError("Escriba una consulta.")
    if contar_tokens(consulta) > modelo.max_seq_length:
        raise ValueError("La consulta supera el límite del modelo.")
    vector = modelo.encode(consulta, normalize_embeddings=True)
    puntajes = matriz @ vector
    return [(registros[int(i)], float(puntajes[i])) for i in np.argsort(-puntajes)[:k]]

In [7]:
#@title Escriba aquí la consulta
CONSULTA = "¿Puedo recibir una constancia si me falta un documento?" #@param {type:"string"}

for nombre in fragmentos:
    print("\nESTRATEGIA:", nombre)
    for registro, puntaje in buscar(CONSULTA, fragmentos[nombre], matrices[nombre]):
        print(f"{registro['source']} · {registro['section']} · coseno {puntaje:.3f}")
        print(registro["chunk_id"])
        print(registro["texto"], "\n")


ESTRATEGIA: ventanas
constancias.md · Constancias académicas · coseno 0.643
constancias:1.0-didactica:ventanas:4
Guía de solicitudes documentales / Constancias académicas
Si falta alguno de los documentos, la solicitud permanece pendiente de información y la constancia no se entrega. Adjuntar el documento faltante permite continuar la revisión, pero no significa que la constancia ya esté emitida.

Conserve el código asignado para consultar el avance. Si necesita corregir los datos de 

constancias.md · Constancias académicas · coseno 0.488
constancias:1.0-didactica:ventanas:3
Guía de solicitudes documentales / Constancias académicas
La solicitud de una constancia académica se radica en el módulo de documentos. Indique el tipo de constancia y el correo al que desea recibirla. Antes de emitirla, se revisan el registro de matrícula y los documentos requeridos para el trámite. Si falta alguno de los documentos, la solicitud permanece pendiente de 

acceso.md · Recuperar contraseña · cosen

Prueben también «Olvidé mi clave y no puedo abrir el correo», «¿Qué hago con un error 403?» y «¿Cómo solicito una beca?». Antes de ejecutar, señalen dónde debería estar la evidencia. Para becas no hay respuesta en este corpus, aunque la búsqueda devuelva candidatos. Anoten qué estrategia conserva mejor la información necesaria en cada caso.

## 6. Guardar y volver a cargar el índice
Elijan una estrategia después de comparar. Guardamos vectores, registros y configuración; luego los leemos nuevamente y repetimos la consulta. Esta colección pequeña utiliza una matriz para búsqueda exacta. En Colab descarguen el ZIP para conservarlo fuera de la sesión.

In [8]:
ESTRATEGIA = "parrafos" #@param ["ventanas", "parrafos"]
destino = Path("indice_clase07")
destino.mkdir(exist_ok=True)
configuracion = {
    "modelo": MODELO, "revision": REVISION,
    "estrategia": ESTRATEGIA, "max_tokens": MAX_TOKENS, "solapamiento": SOLAPAMIENTO,
    "normalizados": True, "similitud": "coseno", "max_seq_length": modelo.max_seq_length,
}
np.save(destino / "vectores.npy", matrices[ESTRATEGIA])
(destino / "registros.json").write_text(json.dumps(fragmentos[ESTRATEGIA], ensure_ascii=False, indent=2), encoding="utf-8")
(destino / "configuracion.json").write_text(json.dumps(configuracion, indent=2), encoding="utf-8")

# Leemos desde disco: no recalculamos los embeddings del corpus.
registros_cargados = json.loads((destino / "registros.json").read_text(encoding="utf-8"))
vectores_cargados = np.load(destino / "vectores.npy", allow_pickle=False)
config_cargada = json.loads((destino / "configuracion.json").read_text(encoding="utf-8"))
assert config_cargada["modelo"] == MODELO and vectores_cargados.shape[0] == len(registros_cargados)
antes = buscar(CONSULTA, fragmentos[ESTRATEGIA], matrices[ESTRATEGIA])
despues = buscar(CONSULTA, registros_cargados, vectores_cargados)
assert [r["chunk_id"] for r, _ in antes] == [r["chunk_id"] for r, _ in despues]
assert np.allclose([p for _, p in antes], [p for _, p in despues])
print("Índice cargado: mismos fragmentos y puntajes para la consulta.")

with zipfile.ZipFile("Clase_07_Indice_y_Corpus.zip", "w", zipfile.ZIP_DEFLATED) as archivo:
    for ruta in [*destino.iterdir(), *carpeta.glob("*.md")]:
        archivo.write(ruta, arcname=ruta.as_posix())
print("Archivo para conservar:", "Clase_07_Indice_y_Corpus.zip")

Índice cargado: mismos fragmentos y puntajes para la consulta.
Archivo para conservar: Clase_07_Indice_y_Corpus.zip


Para usar este índice en otra sesión: descompriman el ZIP, lean la configuración y carguen `SentenceTransformer(config_cargada["modelo"], revision=config_cargada["revision"])`. Lean los registros y la matriz como en la celda anterior. Las consultas deben usar esa misma representación.

In [10]:
#@title Descargar el índice y los documentos (opcional)
DESCARGAR = False #@param {type:"boolean"}
if DESCARGAR:
    from google.colab import files
    files.download("Clase_07_Indice_y_Corpus.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Trabajo del equipo
Reemplacen los dos documentos por fuentes autorizadas del proyecto y ajusten el registro de título, versión y origen. Comparen las dos estrategias con tres preguntas, conserven un resultado útil y un fallo, y guarden el índice elegido. En la bitácora expliquen la decisión y enlacen los documentos, las pruebas y el índice.

**Antes de terminar:** ¿cada fragmento permite volver a la fuente?, ¿se conservan las condiciones del procedimiento?, ¿el índice cargado devuelve los mismos resultados?

Documentación: [modelo de embeddings](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) · [Sentence Transformers](https://sbert.net/docs/package_reference/sentence_transformer/model.html).